# Find similar micro-watersheds within a tehsil

Choose one MWS and find nearby profiles in a standardized hydrology, agriculture, area, and terrain feature space.

Run each cell with **Shift+Enter**. This download is already scoped to the active KYL tehsil and needs no package-installation cell.

In [ ]:
import json, re, sys
from urllib.parse import urlencode
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import geolibre

GEOSERVER_BASE = "https://geoserver.core-stack.org:8443/geoserver/"
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
LAYER_SPECS = json.loads("[{\"id\":\"mws_layers\",\"label\":\"Micro-watersheds and Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_well_depth_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Annual groundwater-storage change and MWS identifiers.\"},{\"id\":\"terrain_vector\",\"label\":\"Terrain Vector\",\"domain\":\"Land\",\"service\":\"WFS\",\"workspace\":\"terrain\",\"layerNameTemplate\":\"{district}_{tehsil}_cluster\",\"period\":\"Current terrain analysis\",\"description\":\"MWS-level plains, slopes, valleys, ridges, hills, and terrain cluster.\"},{\"id\":\"cropping_intensity\",\"label\":\"Cropping Intensity\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"crop_intensity\",\"layerNameTemplate\":\"{district}_{tehsil}_intensity\",\"period\":\"2017 to 2024\",\"description\":\"Annual cropping intensity and single-, double-, and triple-cropped area.\"},{\"id\":\"drought\",\"label\":\"Drought\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"drought\",\"layerNameTemplate\":\"{district}_{tehsil}_drought\",\"period\":\"2017 to 2024\",\"description\":\"Dry spells and weekly mild, moderate, and severe drought indicators.\"}]")
m = geolibre.connect()
MAP_LAYERS = {}

def geoserver_name(value):
    value = re.sub(r"[()]", "", str(value or "").strip().lower())
    return re.sub(r"_+", "_", re.sub(r"\s+", "_", value)).strip("_")

def selected_scope():
    return {
        "state": str(SCOPE["state"]).strip(),
        "district": geoserver_name(SCOPE["district"]),
        "tehsil": geoserver_name(SCOPE["tehsil"]),
    }

def get_spec(layer_id):
    return next(layer for layer in LAYER_SPECS if layer["id"] == layer_id)

def layer_url(layer_id, cql_filter=None, max_features=None):
    scope = selected_scope()
    spec = get_spec(layer_id)
    layer_name = spec["layerNameTemplate"].format(**scope)
    qualified = f'{spec["workspace"]}:{layer_name}'
    if spec["service"] == "WFS":
        params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
                  "typeName": qualified, "outputFormat": "application/json", "srsName": "EPSG:4326"}
        if cql_filter:
            params["CQL_FILTER"] = cql_filter
        if max_features:
            params["maxFeatures"] = int(max_features)
        return f'{GEOSERVER_BASE}{spec["workspace"]}/ows?{urlencode(params)}'
    params = {"service": "WCS", "version": "2.0.1", "request": "GetCoverage",
              "CoverageId": qualified, "format": "geotiff", "compression": "LZW"}
    return f'{GEOSERVER_BASE}{spec["workspace"]}/wcs?{urlencode(params)}'

async def fetch_json(url, label="GeoServer layer"):
    try:
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status}")
            return await response.json()
        import urllib.request
        with urllib.request.urlopen(url, timeout=90) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as error:
        scope = selected_scope()
        raise RuntimeError(
            f'{label} is not available for {scope["district"]}/{scope["tehsil"]}, or GeoServer could not be reached: {error}'
        ) from error

async def load_geojson(layer_id, cql_filter=None, max_features=None):
    spec = get_spec(layer_id)
    if spec["service"] != "WFS":
        raise ValueError(f'{spec["label"]} is a raster. Use its WCS URL instead of loading it as GeoJSON.')
    data = await fetch_json(layer_url(layer_id, cql_filter, max_features), spec["label"])
    if data.get("type") != "FeatureCollection":
        raise RuntimeError(f'{spec["label"]} did not return GeoJSON features.')
    return data

def to_frame(data):
    rows = [dict(feature.get("properties") or {}) for feature in data.get("features", [])]
    return pd.DataFrame(rows)

def uid_column(frame):
    for name in ("uid", "UID", "MWS_UID", "MWS UID"):
        if name in frame.columns:
            return name
    raise KeyError("This layer has no recognised MWS identifier column.")

def with_uid(frame):
    result = frame.copy()
    result["uid"] = result[uid_column(result)].astype(str)
    return result

def numeric(frame, columns):
    return frame.loc[:, columns].apply(pd.to_numeric, errors="coerce")

def json_component(value, component):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        return float(value.get(component)) if isinstance(value, dict) and value.get(component) is not None else np.nan
    except (TypeError, ValueError, json.JSONDecodeError):
        return np.nan

def component_values(frame, columns, component):
    return frame.loc[:, columns].apply(
        lambda series: series.map(lambda value: json_component(value, component))
    )

def features_for_uids(data, uids):
    wanted = {str(uid) for uid in uids}
    names = ("uid", "UID", "MWS_UID", "MWS UID")
    features = []
    for feature in data.get("features", []):
        properties = feature.get("properties") or {}
        value = next((properties.get(name) for name in names if properties.get(name) is not None), None)
        if str(value) in wanted:
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def geojson_bounds(data):
    points = []
    def visit(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            points.append(value[:2])
        elif isinstance(value, list):
            for item in value:
                visit(item)
    for feature in data.get("features", []):
        visit((feature.get("geometry") or {}).get("coordinates", []))
    if not points:
        return None
    xs, ys = zip(*points)
    return [min(xs), min(ys), max(xs), max(ys)]

def show_on_map(key, data, name, **style):
    if not data.get("features"):
        print(f"No features to map for {name}.")
        return None
    previous_layer_id = MAP_LAYERS.get(key)
    if previous_layer_id:
        try:
            m.remove_layer(previous_layer_id)
        except Exception:
            pass
    MAP_LAYERS[key] = m.add_geojson(data, name=name, **style)
    bounds = geojson_bounds(data)
    if bounds:
        m.fit_bounds(bounds)
    return MAP_LAYERS[key]

def year_columns(frame, prefix="", pattern=r"^\d{4}_\d{4}$"):
    return sorted(column for column in frame.columns if column.startswith(prefix) and re.search(pattern, column))

print(f'Ready for {SCOPE["tehsil"]}, {SCOPE["district"]}.')

In [ ]:
PROFILE_METRICS = [
    "Area (ha)", "Mean annual groundwater change", "Recent net groundwater change",
    "Mean cropping intensity", "Mean moderate + severe drought weeks",
    "Plain terrain (%)", "Slope and hill terrain (%)",
]

def build_mws_profile(mws_frame, crop_frame, drought_frame, terrain_frame):
    mws = with_uid(mws_frame).set_index("uid")
    crop = with_uid(crop_frame).set_index("uid")
    drought = with_uid(drought_frame).set_index("uid")
    terrain = with_uid(terrain_frame).set_index("uid")
    common = mws.index.intersection(crop.index).intersection(drought.index).intersection(terrain.index)
    result = pd.DataFrame(index=common)
    area = pd.to_numeric(mws.get("area_in_ha"), errors="coerce").reindex(common)
    groundwater = [column for column in mws.columns if re.match(r"^\d{4}_\d{4}$", column)]
    crop_years = [column for column in crop.columns if re.match(r"^cropping_intensity_\d{4}$", column)]
    drought_years = sorted(set(re.findall(r"\d{4}", " ".join(drought.columns))))
    result["Area (ha)"] = area
    result["Mean annual groundwater change"] = component_values(mws, groundwater, "DeltaG").mean(axis=1).reindex(common)
    result["Recent net groundwater change"] = pd.to_numeric(mws.get("Net2020_25"), errors="coerce").reindex(common)
    result["Mean cropping intensity"] = numeric(crop, crop_years).mean(axis=1).reindex(common)
    drought_values = pd.DataFrame(index=drought.index)
    for year in drought_years:
        moderate = pd.to_numeric(drought.get(f"w_mod_{year}"), errors="coerce")
        severe = pd.to_numeric(drought.get(f"w_sev_{year}"), errors="coerce")
        if moderate is not None and severe is not None:
            drought_values[year] = moderate.add(severe, fill_value=np.nan)
    result["Mean moderate + severe drought weeks"] = drought_values.mean(axis=1).reindex(common)
    plains = pd.to_numeric(terrain.get("plain_area"), errors="coerce").reindex(common)
    slopes = pd.to_numeric(terrain.get("slopy_area"), errors="coerce").reindex(common)
    hills = pd.to_numeric(terrain.get("hill_slope"), errors="coerce").reindex(common)
    result["Plain terrain (%)"] = plains
    result["Slope and hill terrain (%)"] = slopes.add(hills, fill_value=np.nan)
    result.index.name = "MWS UID"
    return result.replace([np.inf, -np.inf], np.nan)

def robust_standardize(profile):
    values = profile[PROFILE_METRICS]
    center = values.median()
    mad = values.sub(center).abs().median()
    iqr = values.quantile(0.75) - values.quantile(0.25)
    scale = (1.4826 * mad).where(mad > 0, iqr / 1.349).replace(0, np.nan)
    return values.sub(center).div(scale), center, scale

def similar_mws(profile, target_uid, count=5):
    standardized, center, scale = robust_standardize(profile)
    target = standardized.loc[str(target_uid)]
    minimum = max(4, int(np.ceil(target.notna().sum() * 0.7)))
    rows = []
    for uid, candidate in standardized.drop(index=str(target_uid)).iterrows():
        shared = target.notna() & candidate.notna()
        if shared.sum() < minimum:
            continue
        distance = np.sqrt(np.mean(np.square(target[shared] - candidate[shared])))
        rows.append({"MWS UID": uid, "Standardized distance": distance,
                     "Compared metrics": int(shared.sum())})
    peers = pd.DataFrame(rows, columns=["MWS UID", "Standardized distance", "Compared metrics"])
    if not peers.empty:
        peers = peers.sort_values(["Standardized distance", "MWS UID"]).head(count)
    return peers, standardized

def plot_similar_profiles(standardized, target_uid, peers):
    order = [str(target_uid)] + peers["MWS UID"].astype(str).tolist()
    values = standardized.loc[order, PROFILE_METRICS]
    fig, ax = plt.subplots(figsize=(12, 5.5))
    image = ax.imshow(values, cmap="PiYG", vmin=-3, vmax=3, aspect="auto")
    ax.set_xticks(range(len(PROFILE_METRICS)), labels=PROFILE_METRICS, rotation=35, ha="right")
    ax.set_yticks(range(len(order)), labels=[f"Target · {order[0]}"] + order[1:])
    ax.set_title("Standardized MWS profiles (clipped color scale at ±3)")
    fig.colorbar(image, ax=ax, label="Robust standardized value")
    plt.tight_layout()
    plt.show()

## 1. Build profiles from four relevant layers

Similarity is calculated only within the selected tehsil and only from comparable published values.

In [ ]:
mws_geojson = await load_geojson("mws_layers")
crop_geojson = await load_geojson("cropping_intensity")
drought_geojson = await load_geojson("drought")
terrain_geojson = await load_geojson("terrain_vector")
profile = build_mws_profile(to_frame(mws_geojson), to_frame(crop_geojson),
                            to_frame(drought_geojson), to_frame(terrain_geojson))
target_uid = sorted(profile.index)[0]
print(f"Using {target_uid} as the example target MWS.")

## 2. Find five similar profiles

Distance is the root-mean-square standardized difference; candidates need at least 70% of the target's available metrics, with a minimum of four.

In [ ]:

peers, standardized = similar_mws(profile, target_uid, count=5)
display(peers.round(3))
plot_similar_profiles(standardized, target_uid, peers)

## 3. Compare their locations

The target and its similar profiles are separate temporary layers so their roles remain clear.

In [ ]:
target_features = features_for_uids(mws_geojson, [target_uid])
peer_features = features_for_uids(mws_geojson, peers["MWS UID"])
show_on_map("similar-target", target_features, f"Notebook target · {target_uid}",
            fillColor="#facc15", strokeColor="#713f12", fillOpacity=0.8)
show_on_map("similar-peers", peer_features, "Notebook · similar MWS profiles",
            fillColor="#22c55e", strokeColor="#14532d", fillOpacity=0.58)

## Interpretation

These are similar data profiles, not necessarily geographic neighbours or interchangeable communities. Changing the variables, period, missing-data rule, or distance definition can change the result.

## Optional: compare raw values

In [ ]:
comparison_ids = [str(target_uid)] + peers["MWS UID"].astype(str).tolist()
display(profile.loc[comparison_ids, PROFILE_METRICS].round(2))